# Interior valid-zeros


interior valid-zeros: rows where the status column says valid but the value is 0,
positioned between the first and last genuine reading of that series.

Definitions used here:
- genuine reading = status is valid`AND value ≠ 0 (a real physiological measurement),
- interior valid-zero = status valid, value 0, lying strictly inside the span between
  the first and last genuine reading (so leading/trailing gaps are excluded).


## Setup & configuration

In [ ]:
import os, re, zipfile
import numpy as np, pandas as pd
from IPython.display import display

SUMMARY_CSV = "summary.csv"
ZIP_PATH    = "CSVFiles.zip"
ID_COL      = "Filename"
TIME_COL    = "datetime"
OUTPUT_DIR  = "eda_outputs"

MEASURES = {
    "Hr":       {"token": "hr",    "summary_col": "Hr",    "value_col": "HR_value",        "vmin": 20, "vmax": 220},
    "Spo2":     {"token": "spo2",  "summary_col": "Spo2",  "value_col": "Spo2_value",      "vmin": 40, "vmax": 100},
    "Nibp_sys": {"token": "nibp",  "summary_col": "Nibp",  "value_col": "Nibp_Sys_value",  "vmin": 50, "vmax": 250},
    "Nibp_dia": {"token": "nibp",  "summary_col": "Nibp",  "value_col": "Nibp_Dia_value",  "vmin": 20, "vmax": 150},
    "Nibp_mean":{"token": "nibp",  "summary_col": "Nibp",  "value_col": "Nibp_mean_value", "vmin": 30, "vmax": 180},
    "Ibp_sys":  {"token": "ibp",   "summary_col": "Ibp",   "value_col": "ibp0_Sys_value",  "vmin": 50, "vmax": 250},
    "Ibp_dia":  {"token": "ibp",   "summary_col": "Ibp",   "value_col": "ibp0_Dia_value",  "vmin": 20, "vmax": 150},
    "Ibp_map":  {"token": "ibp",   "summary_col": "Ibp",   "value_col": "ibp0_Mean_value", "vmin": 30, "vmax": 180},
    "Etco2":    {"token": "etco2", "summary_col": "Etco2", "value_col": "Etco2_value",     "vmin": 5,  "vmax": 80},
    "Resp":     {"token": "resp",  "summary_col": "Resp",  "value_col": "Resp_value",      "vmin": 3,  "vmax": 60},
    "Temp":     {"token": "temp",  "summary_col": "Temp",  "value_col": "Temp0_value", "scale": 10, "vmin": 25, "vmax": 43},
    "SpPI":     {"token": "spi",   "summary_col": "SpPI",  "value_col": "PI_value",        "vmin": 0,  "vmax": 100},
    "SpPVI":    {"token": "spvi",  "summary_col": "SpPVI", "value_col": "PVI_value",       "vmin": 0,  "vmax": 100},
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
df = pd.read_csv(SUMMARY_CSV); df.columns = [c.strip() for c in df.columns]
zf = zipfile.ZipFile(ZIP_PATH); names = set(zf.namelist())
pd.set_option("display.max_columns", None); pd.set_option("display.width", 140)
print("recordings in summary:", len(df))

recordings in summary: 606


## Analysis function


In [ ]:
def detail_name(filename, token):
    fn = str(filename).lower()
    for n in names:
        base = n.lower()
        if fn in base and base.endswith(".csv") and re.search(rf"(^|[_/]){token}([_.])", base):
            return n
    return None

# only keep the files where the number of per-channel recordings is > 0
def analyze_measure(measure):
    cfg = MEASURES[measure]
    counts = pd.to_numeric(df[cfg["summary_col"]], errors="coerce").fillna(0)
    recordings = df.loc[counts > 0, ID_COL].tolist()

    n_files = n_affected = total_interior = total_valid = 0
    per_record = []
    # loop over each qualifying file and find its detail file in zip
    for fn in recordings:
        name = detail_name(fn, cfg["token"])
        if name is None:
            continue
        with zf.open(name) as f:
            d = pd.read_csv(f)
        d.columns = [c.strip() for c in d.columns]

        vcol = cfg["value_col"]

        if TIME_COL in d.columns:
            d = d.sort_values(TIME_COL).reset_index(drop=True)

        # status = column directly left of value
        vpos = d.columns.get_loc(vcol)
        if vpos == 0:
            continue
        scol = d.columns[vpos - 1]

        n_files += 1
        vals  = pd.to_numeric(d[vcol], errors="coerce").to_numpy() / cfg.get("scale", 1)
        valid = d[scol].astype(str).str.strip().str.lower().eq("valid").to_numpy()
        total_valid += int(valid.sum())

        genuine = valid & (vals != 0) & ~np.isnan(vals)      # real physiological reading
        gidx = np.where(genuine)[0]
        if len(gidx) < 2:
            per_record.append((fn, 0)); continue
        first_real, last_real = gidx.min(), gidx.max()

        interior = valid & (vals == 0)                        # valid status, zero value
        interior[:first_real + 1] = False                     # exclude up to first genuine
        interior[last_real:]      = False                     # exclude from last genuine
        c = int(interior.sum())
        total_interior += c
        if c > 0:
            n_affected += 1
        per_record.append((fn, c))

    #summary statistics for each measure
    pct_rows = (total_interior / total_valid * 100) if total_valid else 0.0
    stats = {
        "measure": measure,
        "n_recordings": n_files,
        "recordings_affected": n_affected,
        "pct_recordings_affected": round(n_affected / n_files * 100, 1) if n_files else 0.0,
        "total_interior_valid_zeros": total_interior,
        "total_valid_rows": total_valid,
        "pct_of_valid_rows": round(pct_rows, 3),
    }
    return stats, per_record

## Runs across all channels

In [ ]:
summary_rows, details = [], {}
for m in MEASURES:
    stats, per_record = analyze_measure(m)
    summary_rows.append(stats)
    details[m] = per_record
    print(f"[{m:9s}] {stats['n_recordings']:>4} files | "
          f"{stats['recordings_affected']:>3} affected | "
          f"{stats['total_interior_valid_zeros']:>5} interior valid-zeros "
          f"({stats['pct_of_valid_rows']}% of valid rows)")

result = pd.DataFrame(summary_rows).sort_values("total_interior_valid_zeros", ascending=False)
result.to_csv(os.path.join(OUTPUT_DIR, "interior_valid_zeros.csv"), index=False)
display(result)

[Hr       ]  605 files |   0 affected |     0 interior valid-zeros (0.0% of valid rows)
[Spo2     ]  595 files |   0 affected |     0 interior valid-zeros (0.0% of valid rows)
[Nibp_sys ]  595 files |   0 affected |     0 interior valid-zeros (0.0% of valid rows)
[Nibp_dia ]  595 files |   1 affected |     4 interior valid-zeros (0.01% of valid rows)
[Nibp_mean]  595 files |   0 affected |     0 interior valid-zeros (0.0% of valid rows)
[Ibp_sys  ]  172 files |  19 affected |   209 interior valid-zeros (1.336% of valid rows)
[Ibp_dia  ]  172 files |  27 affected |   167 interior valid-zeros (1.074% of valid rows)
[Ibp_map  ]  172 files |  25 affected |   180 interior valid-zeros (1.151% of valid rows)
[Etco2    ]  421 files | 147 affected |  1491 interior valid-zeros (3.148% of valid rows)
[Resp     ]  559 files |   0 affected |     0 interior valid-zeros (0.0% of valid rows)
[Temp     ]    9 files |   0 affected |     0 interior valid-zeros (0.0% of valid rows)
[SpPI     ]    0 files 

,measure,n_recordings,recordings_affected,pct_recordings_affected,total_interior_valid_zeros,total_valid_rows,pct_of_valid_rows
8,Etco2,421,147,34.9,1491,47363,3.148
5,Ibp_sys,172,19,11.0,209,15648,1.336
7,Ibp_map,172,25,14.5,180,15642,1.151
6,Ibp_dia,172,27,15.7,167,15548,1.074
3,Nibp_dia,595,1,0.2,4,39240,0.010
4,Nibp_mean,595,0,0.0,0,38826,0.000
2,Nibp_sys,595,0,0.0,0,39257,0.000
1,Spo2,595,0,0.0,0,55402,0.000
0,Hr,605,0,0.0,0,74966,0.000
9,Resp,559,0,0.0,0,57174,0.000


## Drill-down


In [ ]:
m = "Etco2"
pr = pd.DataFrame(details[m], columns=["Filename", "interior_valid_zeros"])
print(f"{m}: {(pr['interior_valid_zeros'] > 0).sum()} of {len(pr)} recordings affected")
display(pr.sort_values("interior_valid_zeros", ascending=False).head(10))

Etco2: 147 of 421 recordings affected


,Filename,interior_valid_zeros
256,260618_3339_Deid_JSON_359,71
380,260618_3728_Deid_JSON_531,62
128,260618_3206_Deid_JSON_450,52
225,260618_3646_Deid_JSON_438,51
156,260618_3421_Deid_JSON_505,39
312,260618_3438_Deid_JSON_456,37
315,260415_4112_Deid_JSON_025,36
314,260618_3918_Deid_JSON_412,32
76,260415_3853_Deid_JSON_024,30
219,260618_3327_Deid_JSON_461,29


## Join interior-zero flags to patient outcomes

Builds, for each patient, whether their EtCO₂ / IBP / NIBP channel showed interior valid-zeros,
and joins to the outcome table. Restricted throughout to patients who have the channel, so
the comparison is affected vs unaffected within channel users (not the "sicker patients get the
channel" confound).

In [ ]:
from scipy.stats import mannwhitneyu, fisher_exact

OUTCOME_PATH = "Deidentified_EPCR_ZOLL_London_Trauma_ZOLL_03_04.xlsx"
OUTCOME_ID   = "Deidentified_JSON_ZOLL_x"

# channel groups: presence column in summary + the detail measures that make it up
CHANNEL_GROUPS = {
    "Etco2": {"summary_col": "Etco2", "measures": ["Etco2"]},
    "Ibp":   {"summary_col": "Ibp",   "measures": ["Ibp_sys", "Ibp_dia", "Ibp_map"]},
    "Nibp":  {"summary_col": "Nibp",  "measures": ["Nibp_sys", "Nibp_dia", "Nibp_mean"]},
}
OUTCOMES = ["Age_x", "First GCS", "First lactate (hospital arrival)", "24hr Prehospital + Hospital Blood Transfusion",
            "Outcome Alive 0/Dead 1", "CODE RED",
            "Days Survived", "Scene Arrive to Hospital time"]

def load_outcomes(path):
    o = pd.read_excel(path) if path.lower().endswith((".xlsx", ".xls")) else pd.read_csv(path)
    o.columns = [c.strip() for c in o.columns]
    return o
outc = load_outcomes(OUTCOME_PATH)

# build per-patient affected flags from the already-computed `details`
patient = pd.DataFrame({ID_COL: df[ID_COL].values})
for g, spec in CHANNEL_GROUPS.items():
    has = pd.to_numeric(df[spec["summary_col"]], errors="coerce").fillna(0) > 0
    patient[f"has_{g}"] = has.values
    total = pd.Series(0.0, index=df[ID_COL].values)
    for m in spec["measures"]:
        if m in details:
            total = total.add(pd.Series(dict(details[m])), fill_value=0)
    patient[f"izeros_{g}"]   = patient[ID_COL].map(total).fillna(0).astype(int)
    patient[f"affected_{g}"] = patient[f"izeros_{g}"] > 0
    print(f"[{g}] have channel: {int(has.sum())} | affected: {int(patient[f'affected_{g}'].sum())}")

merged = patient.merge(outc, left_on=ID_COL, right_on=OUTCOME_ID, how="left")
matched = merged[OUTCOME_ID].notna().sum()
print(f"\njoined to outcomes: {matched} of {len(patient)} matched ({matched/len(patient)*100:.1f}%)")

[Etco2] have channel: 421 | affected: 147
[Ibp] have channel: 172 | affected: 42
[Nibp] have channel: 595 | affected: 1

joined to outcomes: 606 of 606 matched (100.0%)


## Descriptive clinical profile (affected vs unaffected)
rates and medians for patients whose channel showed interior
valid-zeros, vs those whose channel did not, among patients who have the channel.

In [ ]:
# clinical descriptors to profile
PROFILE_BINARY  = ["Outcome Alive 0/Dead 1", "CODE RED", "RSI", "Thoracostomy", "TXA"]
PROFILE_NUMERIC = ["Head AIS_x", "ISS_x", "First GCS", "Scene Arrive to Hospital time",
                   "24hr Prehospital + Hospital Blood Transfusion"]

# flag any columns not present so the names can be corrected
missing = [x for x in PROFILE_BINARY + PROFILE_NUMERIC if x not in merged.columns]
if missing:
    print("NOT FOUND (fix these names):", missing)
    print("head/injury-ish columns present:",
          [x for x in merged.columns if any(k in x.lower() for k in ["ais", "iss", "gcs", "head"])])

# affected vs unaffected profile for one channel group, restricted to patients who HAVE it
def profile(group):
    sub = merged[merged[f"has_{group}"] == True].copy()
    aff = sub[f"affected_{group}"].astype(bool)
    print(f"\n===== {group}: affected {int(aff.sum())}  vs  unaffected {int((~aff).sum())} "
          f"(among {len(sub)} who have {group}) =====")
    rows = []
    # binary descriptors -> % and count
    for v in PROFILE_BINARY:
        if v not in sub.columns: continue
        num = pd.to_numeric(sub[v], errors="coerce")
        a, b = num[aff].dropna(), num[~aff].dropna()
        if len(a) and len(b):
            rows.append({"descriptor": v,
                         "affected":   f"{a.mean()*100:.0f}%  ({int(a.sum())}/{len(a)})",
                         "unaffected": f"{b.mean()*100:.0f}%  ({int(b.sum())}/{len(b)})"})
    # numeric descriptors -> median [IQR]
    for v in PROFILE_NUMERIC:
        if v not in sub.columns: continue
        num = pd.to_numeric(sub[v], errors="coerce")
        a, b = num[aff].dropna(), num[~aff].dropna()
        if len(a) and len(b):
            fmt = lambda s: f"{s.median():.1f}  [{s.quantile(.25):.1f}-{s.quantile(.75):.1f}]"
            rows.append({"descriptor": v, "affected": fmt(a), "unaffected": fmt(b)})
    return pd.DataFrame(rows)

for g in ["Etco2", "Ibp"]:
    display(profile(g))


===== Etco2: affected 147  vs  unaffected 274 (among 421 who have Etco2) =====


,descriptor,affected,unaffected
0,Outcome Alive 0/Dead 1,30% (42/138),16% (41/252)
1,CODE RED,63% (93/147),64% (176/274)
2,RSI,50% (74/147),34% (93/274)
3,Thoracostomy,37% (55/147),26% (70/274)
4,TXA,65% (95/147),70% (191/274)
5,Head AIS_x,1.0 [0.0-4.0],0.0 [0.0-3.0]
6,ISS_x,26.0 [16.5-49.0],29.0 [20.0-42.5]
7,First GCS,13.0 [10.0-15.0],14.0 [6.0-15.0]
8,Scene Arrive to Hospital time,50.0 [35.5-67.0],44.0 [27.0-60.8]
9,24hr Prehospital + Hospital Blood Transfusion,4.0 [2.0-8.0],3.0 [1.0-6.0]



===== Ibp: affected 42  vs  unaffected 130 (among 172 who have Ibp) =====


,descriptor,affected,unaffected
0,Outcome Alive 0/Dead 1,31% (13/42),31% (38/124)
1,CODE RED,64% (27/42),63% (82/130)
2,RSI,62% (26/42),46% (60/130)
3,Thoracostomy,26% (11/42),41% (53/130)
4,TXA,60% (25/42),69% (90/130)
5,Head AIS_x,2.0 [0.0-5.0],1.0 [0.0-4.0]
6,ISS_x,32.0 [26.0-48.5],43.0 [26.0-57.0]
7,Scene Arrive to Hospital time,62.0 [46.5-71.5],53.0 [41.0-65.8]
8,24hr Prehospital + Hospital Blood Transfusion,4.0 [2.0-8.0],4.0 [2.0-8.8]


### Head-injury focus (EtCO₂)


In [ ]:
HEAD_COL = "Head AIS_x"

sub = merged[merged["has_Etco2"] == True].copy()
aff = sub["affected_Etco2"].astype(bool)
ais = pd.to_numeric(sub[HEAD_COL], errors="coerce")
print(f"Head AIS available for {ais.notna().sum()} of {len(sub)} EtCO2 patients\n")

# (a) significant head injury (AIS >= 3) in affected vs unaffected
sig = (ais >= 3)
for label, mask in [("affected", aff), ("unaffected", ~aff)]:
    m = mask & ais.notna()
    print(f"{label:11s}: significant head injury (AIS>=3) in "
          f"{sig[m].mean()*100:.0f}%  ({int(sig[m].sum())}/{int(m.sum())})")

# (b) interior-zero affectedness across head-injury severity bands
print("\ninterior EtCO2 zeros by head-injury severity:")
bands = pd.cut(ais, [-0.1, 0.9, 2.9, 4.9, 6.1], labels=["AIS 0", "AIS 1-2", "AIS 3-4", "AIS 5-6"])
tab = pd.DataFrame({"band": bands, "affected": aff}).dropna(subset=["band"])
summary = tab.groupby("band", observed=True)["affected"].agg(n="size", n_affected="sum")
summary["% affected"] = (summary["n_affected"] / summary["n"] * 100).round(0)
display(summary)

Head AIS available for 416 of 421 EtCO2 patients

affected   : significant head injury (AIS>=3) in 40%  (57/143)
unaffected : significant head injury (AIS>=3) in 28%  (76/273)

interior EtCO2 zeros by head-injury severity:


,n,n_affected,% affected
band,,,
AIS 0,224,65,29.0
AIS 1-2,59,21,36.0
AIS 3-4,67,28,42.0
AIS 5-6,65,29,45.0
